In [2]:
import diplotocus as dpl
import matplotlib.pyplot as plt
import numpy as np

import logging
from types import ModuleType
from importlib import reload, import_module
from IPython.core.magic import register_line_magic, register_cell_magic

logger = logging.getLogger(__name__)

def _reload(module, reload_all, reloaded):
    if isinstance(module, ModuleType):
        module_name = module.__name__
    elif isinstance(module, str):
        module_name, module = module, import_module(module)
    else:
        raise TypeError(
            "'module' must be either a module or str; "
            f"got: {module.__class__.__name__}")

    for attr_name in dir(module):
        attr = getattr(module, attr_name)
        check = (
            # is it a module?
            isinstance(attr, ModuleType)

            # has it already been reloaded?
            and attr.__name__ not in reloaded

            # is it a proper submodule? (or just reload all)
            and (reload_all or attr.__name__.startswith(module_name))
        )
        if check:
            _reload(attr, reload_all, reloaded)

    logger.debug(f"reloading module: {module.__name__}")
    reload(module)
    reloaded.add(module_name)


def reload_recursive(module, reload_external_modules=False):
    _reload(module, reload_external_modules, set())


@register_line_magic('reload')
def reload_magic(module):
    reload_recursive(module)

In [20]:
reload_magic(dpl)

x = np.linspace(0,6*np.pi,200)
y = np.cos(x)

seq = dpl.Sequence()
seq.animate(dpl.scatter(x=x,y=y,duration=25).spawn())
seq.save_video()

  0%|          | 0/25 [00:00<?, ?it/s]

Saved video Unnamed.mp4